# Сравнение XGBoost, CatBoost и LightGBM

Цель — сравнить три градиентных бустинга для прогноза `rented_bike_count` на Seoul Bike Sharing. Используются те же признаки и хронологическая схема разбиения, что и в `baseline.ipynb`: 70% train, 15% validation и последние 15% test. Test остаётся отложенным и не используется при выборе модели.

Сравнение включает expanding `TimeSeriesSplit` на train и итоговые метрики на одном validation-периоде. Масштабирование числовых признаков не требуется.

## 1. Импорты и параметры

In [1]:
import os
import re
from copy import deepcopy

import kagglehub
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

RANDOM_STATE = 42
N_SPLITS = 5

## 2. Загрузка и подготовка данных

Названия столбцов приводятся к тем же именам, что используются в baseline. Данные сортируются по дате и часу до разбиения.

In [2]:
CSV_PATH = os.path.join(
    kagglehub.dataset_download("joebeachcapital/seoul-bike-sharing"),
    "SeoulBikeData.csv",
)
df = pd.read_csv(CSV_PATH, encoding="unicode_escape")


def norm_col(column: str) -> str:
    column = str(column).strip().lower()
    column = column.replace(chr(176) + "c", "").replace(chr(176), "")
    column = column.replace("%", "")
    column = column.replace("(", "").replace(")", "")
    column = column.replace("/", "_")
    column = re.sub(r"\s+", "_", column)
    return re.sub(r"_+", "_", column).strip("_")


df.columns = [norm_col(column) for column in df.columns]
df = df.rename(
    columns={
        "wind_speed_m_s": "wind_speed",
        "visibility_10m": "visibility",
        "dew_point_temperature": "dew_point_temp",
        "solar_radiation_mj_m2": "solar_radiation",
        "rainfallmm": "rainfall",
        "snowfall_cm": "snowfall",
    }
)
df["date"] = pd.to_datetime(df["date"], format="%d/%m/%Y")
df = df.sort_values(["date", "hour"]).reset_index(drop=True)

print(f"Размер датасета: {df.shape}")
print(f"Период: {df['date'].min():%Y-%m-%d} — {df['date'].max():%Y-%m-%d}")

Размер датасета: (8760, 14)
Период: 2017-12-01 — 2018-11-30


## 3. Хронологическое разбиение

Границы рассчитаны так, чтобы сохранить разбиение baseline (6132 / 1314 / 1314 для 8760 строк). Последние 15% данных не участвуют ни в кросс-валидации, ни в сравнении моделей.

In [3]:
test_size = int(len(df) * 0.15)
validation_size = int(len(df) * 0.15)
train_size = len(df) - validation_size - test_size

train_df = df.iloc[:train_size].copy()
validation_df = df.iloc[train_size : train_size + validation_size].copy()
test_df = df.iloc[train_size + validation_size :].copy()

target = "rented_bike_count"
features = [column for column in df.columns if column not in ["date", target]]
categorical_features = ["seasons", "holiday", "functioning_day"]
numeric_features = [column for column in features if column not in categorical_features]

X_train = train_df[features]
y_train = train_df[target]
X_validation = validation_df[features]
y_validation = validation_df[target]

print(
    f"Train: {len(train_df)}, validation: {len(validation_df)}, "
    f"test (отложен): {len(test_df)}"
)

Train: 6132, validation: 1314, test (отложен): 1314


## 4. Препроцессинг и модели

XGBoost и LightGBM получают One-Hot-кодирование категорий; известные уровни заданы явно, чтобы категория сезона, отсутствующая в train, не теряла свой признак на validation. CatBoost получает исходные категории и обрабатывает их нативно. Для всех моделей используются одни и те же строки и целевая переменная.

In [4]:
category_levels = [
    ["Autumn", "Spring", "Summer", "Winter"],
    ["Holiday", "No Holiday"],
    ["No", "Yes"],
]
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                categories=category_levels,
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_features,
        ),
        ("numeric", "passthrough", numeric_features),
    ]
)

models = {
    "XGBoost": make_pipeline(
        preprocessor,
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            eval_metric="rmse",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0,
        ),
    ),
    "LightGBM": make_pipeline(
        preprocessor,
        LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1,
        ),
    ),
    "CatBoost": CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        cat_features=categorical_features,
        random_seed=RANDOM_STATE,
        verbose=0,
        allow_writing_files=False,
        thread_count=-1,
    ),
}

print("Модели подготовлены:", ", ".join(models))

Модели подготовлены: XGBoost, LightGBM, CatBoost


## 5. Кросс-валидация и сравнение на validation

Каждый fold использует только более ранние наблюдения для обучения. Для каждого алгоритма после CV выполняется одно обучение на всём train и одна оценка на validation. Test не трогаем.

In [5]:
time_series_cv = TimeSeriesSplit(n_splits=N_SPLITS)
results = []
fitted_models = {}

for model_name, estimator in models.items():
    fold_rmse = []

    for train_indices, fold_indices in time_series_cv.split(X_train):
        fold_model = deepcopy(estimator)
        X_fold_train = X_train.iloc[train_indices]
        y_fold_train = y_train.iloc[train_indices]
        X_fold_validation = X_train.iloc[fold_indices]
        y_fold_validation = y_train.iloc[fold_indices]

        fold_model.fit(X_fold_train, y_fold_train)
        fold_predictions = fold_model.predict(X_fold_validation)
        fold_rmse.append(
            np.sqrt(mean_squared_error(y_fold_validation, fold_predictions))
        )

    final_model = deepcopy(estimator)
    final_model.fit(X_train, y_train)
    validation_predictions = final_model.predict(X_validation)
    fitted_models[model_name] = final_model

    results.append(
        {
            "model": model_name,
            "cv_rmse_mean": np.mean(fold_rmse),
            "cv_rmse_std": np.std(fold_rmse),
            "validation_rmse": np.sqrt(
                mean_squared_error(y_validation, validation_predictions)
            ),
            "validation_mae": mean_absolute_error(
                y_validation, validation_predictions
            ),
            "validation_r2": r2_score(y_validation, validation_predictions),
        }
    )

    print(f"{model_name}: CV RMSE = {np.mean(fold_rmse):.2f} "
          f"+/- {np.std(fold_rmse):.2f}")

results_df = pd.DataFrame(results).sort_values("validation_rmse").reset_index(drop=True)
results_df.round(3)

XGBoost: CV RMSE = 335.82 +/- 122.02
LightGBM: CV RMSE = 349.64 +/- 138.02
CatBoost: CV RMSE = 335.14 +/- 124.58


,model,cv_rmse_mean,cv_rmse_std,validation_rmse,validation_mae,validation_r2
0,CatBoost,335.141,124.580,350.463,250.008,0.760
1,XGBoost,335.818,122.019,354.833,245.727,0.754
2,LightGBM,349.640,138.018,412.123,272.912,0.668


## 6. Результаты

| Модель | CV RMSE, mean ± std | Validation RMSE | Validation MAE | Validation R² |
| --- | ---: | ---: | ---: | ---: |
| CatBoost | 335.14 ± 124.58 | **350.46** | 250.01 | 0.7597 |
| XGBoost | 335.82 ± 122.02 | 354.83 | **245.73** | 0.7537 |
| LightGBM | 349.64 ± 138.02 | 412.12 | 272.91 | 0.6677 |

С начальными параметрами CatBoost лидирует по RMSE и R², XGBoost близок и показывает немного меньший MAE. LightGBM на этом разбиении заметно уступает. Это сравнение без подбора гиперпараметров; test остаётся нетронутым.

## 7. Итог

Ниже код повторно выводит лидера по validation RMSE. Перед финальной фиксацией лучшую модель нужно переобучить на train + validation и один раз оценить на отложенном test.

In [6]:
best_result = results_df.iloc[0]
print(
    f"Лучшая по validation RMSE модель: {best_result['model']} "
    f"(RMSE={best_result['validation_rmse']:.2f}, "
    f"R2={best_result['validation_r2']:.4f})."
)
print("Test-метрики не рассчитаны и не использовались для выбора.")

Лучшая по validation RMSE модель: CatBoost (RMSE=350.46, R2=0.7597).
Test-метрики не рассчитаны и не использовались для выбора.
